# Moment Flow vs DSEC Ground Truth Analysis

Use this notebook after running `event_detector_cpp` with raw-flow saving enabled. Each run writes **two** prediction sets under `flow_save_output_dir`:

- `dense/` — `flow_dense`, the full dense MomentFlow field (every pixel valid), benchmarked with `--mask-mode gt`.
- `sparse/` — `flow_events`, the dense field masked to event-supported pixels (unsupported → `valid=0`), benchmarked with `--mask-mode intersection`.

The notebook runs both benchmarks, compares them side by side, then drops into a per-frame deep dive on whichever variant `ACTIVE` selects. The goal is not just to report the score, but to isolate likely failure modes: timestamp alignment, DSEC PNG channel order, sign conventions, scale errors, spatially localized failures, and outlier frames.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find the workspace root whether the notebook is opened from the root or this folder.
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src/event_detector_cpp").exists() and (candidate / "tools/dsec_flow_benchmark").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT))

from tools.dsec_flow_benchmark.flow_io import load_dsec_png
from tools.dsec_flow_benchmark.timestamps import load_timestamps

GT_DIR = ROOT / "logs/dsec/thun_00_a/optical_flow_forward"
TIMESTAMP_FILE = GT_DIR / "thun_00_a_optical_flow_forward_timestamps.txt"

# event_detector_cpp now saves two prediction sets per run under flow_save_output_dir:
#   dense/  -> flow_dense  (full dense field, every pixel valid)        -> mask-mode gt
#   sparse/ -> flow_events (dense flow masked to event support, NaN->0) -> mask-mode intersection
BASE_PRED_DIR = ROOT / "logs/moment_flow/thun_00_a"

VARIANTS = {
    "dense": {
        "pred_dir": BASE_PRED_DIR / "dense",
        "mask_mode": "gt",
    },
    "sparse": {
        "pred_dir": BASE_PRED_DIR / "sparse",
        "mask_mode": "intersection",
    },
}
for cfg in VARIANTS.values():
    cfg["result_json"] = cfg["pred_dir"] / "benchmark.json"
    cfg["result_csv"] = cfg["pred_dir"] / "benchmark_frames.csv"

# Which variant the per-frame deep-dive cells (worst frames, sign/scale, spatial bias,
# correlation) inspect. Set to "dense" or "sparse" and rerun from the load cell down.
ACTIVE = "sparse"

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})


def png_names(directory):
    return sorted(path.name for path in Path(directory).glob("*.png"))


def read_result_frame_names(result_csv):
    if not Path(result_csv).exists():
        return []
    try:
        frame_df = pd.read_csv(result_csv)
    except Exception:
        return []
    if "frame" not in frame_df.columns:
        return []
    return sorted(frame_df["frame"].astype(str).tolist())


def benchmark_current_status(pred_dir, result_json, result_csv):
    gt_names = png_names(GT_DIR)
    pred_names = png_names(pred_dir)
    expected = sorted(set(gt_names) & set(pred_names))
    csv_frames = read_result_frame_names(result_csv)
    reasons = []

    if not gt_names:
        reasons.append(f"No ground-truth PNGs found in {GT_DIR}")
    if not pred_names:
        reasons.append(f"No prediction PNGs found in {pred_dir}")
    if not Path(result_json).exists() or not Path(result_csv).exists():
        reasons.append("Benchmark metric files are missing")

    stale_csv = sorted(name for name in csv_frames if name not in gt_names or name not in pred_names)
    missing_csv = sorted(name for name in expected if name not in set(csv_frames))
    if Path(result_csv).exists() and not csv_frames:
        reasons.append("Benchmark CSV exists but has no readable frame rows")
    if stale_csv:
        reasons.append(f"Benchmark CSV references {len(stale_csv)} frame(s) not present on disk")
    if missing_csv:
        reasons.append(f"Benchmark CSV is missing {len(missing_csv)} current prediction frame(s)")

    current = (
        bool(expected)
        and Path(result_json).exists()
        and Path(result_csv).exists()
        and not stale_csv
        and not missing_csv
    )
    return {
        "current": current,
        "can_run": bool(expected),
        "gt_names": gt_names,
        "pred_names": pred_names,
        "expected": expected,
        "csv_frames": csv_frames,
        "stale_csv": stale_csv,
        "missing_csv": missing_csv,
        "reasons": reasons,
    }


print("ROOT:", ROOT)
print("GT_DIR:", GT_DIR)
print("ACTIVE variant:", ACTIVE)
for name, cfg in VARIANTS.items():
    print(f"  {name:6s} pred_dir={cfg['pred_dir']}  mask_mode={cfg['mask_mode']}  "
          f"result_json_exists={cfg['result_json'].exists()}")

## Run Or Refresh The Benchmarks

Runs both the dense and sparse benchmarks, reusing existing `benchmark.json`/`benchmark_frames.csv` when current and regenerating them otherwise (set `FORCE_REFRESH = True` to force). The `dense` variant scores with `--mask-mode gt`; the `sparse` variant with `--mask-mode intersection` so only event-supported, DSEC-valid pixels count.

In [ ]:
FORCE_REFRESH = False


def run_or_refresh(name, cfg, force_refresh=False):
    status = benchmark_current_status(cfg["pred_dir"], cfg["result_json"], cfg["result_csv"])
    should_run = (force_refresh or not status["current"]) and status["can_run"]
    cmd = [
        sys.executable, "-m", "tools.dsec_flow_benchmark",
        "--gt-dir", str(GT_DIR),
        "--pred-dir", str(cfg["pred_dir"]),
        "--match", "name",
        "--mask-mode", cfg["mask_mode"],
        "--allow-missing",
        "--output-json", str(cfg["result_json"]),
        "--output-csv", str(cfg["result_csv"]),
    ]
    if should_run:
        print(f"[{name}] Running:", " ".join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)
    elif status["current"]:
        print(f"[{name}] Using current benchmark files. Set FORCE_REFRESH=True to refresh.")
    else:
        print(f"[{name}] Cannot run yet: no GT/prediction PNG name overlap.")
    final = benchmark_current_status(cfg["pred_dir"], cfg["result_json"], cfg["result_csv"])
    print(f"[{name}] ready={final['current']}  GT={len(final['gt_names'])}  "
          f"Pred={len(final['pred_names'])}  matched={len(final['expected'])}")
    for reason in final["reasons"]:
        print(f"  - {reason}")
    return final


VARIANT_STATUS = {name: run_or_refresh(name, cfg, FORCE_REFRESH) for name, cfg in VARIANTS.items()}

# Bind the active variant so the per-frame deep-dive cells below operate on one set.
PRED_DIR = VARIANTS[ACTIVE]["pred_dir"]
RESULT_JSON = VARIANTS[ACTIVE]["result_json"]
RESULT_CSV = VARIANTS[ACTIVE]["result_csv"]
MASK_MODE = VARIANTS[ACTIVE]["mask_mode"]
BENCHMARK_STATUS = VARIANT_STATUS[ACTIVE]
ANALYSIS_READY = BENCHMARK_STATUS["current"]
print("\nActive variant for deep-dive:", ACTIVE, "->", PRED_DIR)
print("Analysis ready:", ANALYSIS_READY)

In [ ]:
timing = load_timestamps(TIMESTAMP_FILE)
timing_by_frame = {
    path.name: row for path, row in zip(sorted(GT_DIR.glob("*.png")), timing)
}


def load_results(cfg):
    status = benchmark_current_status(cfg["pred_dir"], cfg["result_json"], cfg["result_csv"])
    if not status["current"]:
        return {"summary": {}}, pd.DataFrame()
    with Path(cfg["result_json"]).open("r", encoding="utf-8") as f:
        benchmark = json.load(f)
    frame_df = pd.read_csv(cfg["result_csv"])
    frame_df["frame_index"] = frame_df["frame"].str.replace(".png", "", regex=False).astype(int)
    frame_df = frame_df.sort_values("frame_index").reset_index(drop=True)
    frame_df["dt_ms"] = frame_df["frame"].map(
        lambda name: timing_by_frame[name].dt_us / 1000.0 if name in timing_by_frame else np.nan)
    frame_df["from_us"] = frame_df["frame"].map(
        lambda name: timing_by_frame[name].from_us if name in timing_by_frame else np.nan)
    frame_df["to_us"] = frame_df["frame"].map(
        lambda name: timing_by_frame[name].to_us if name in timing_by_frame else np.nan)
    return benchmark, frame_df


VARIANT_RESULTS = {}
for name, cfg in VARIANTS.items():
    bench, frame_df = load_results(cfg)
    VARIANT_RESULTS[name] = {"benchmark": bench, "df": frame_df}

# Side-by-side summary of both benchmarks.
summary_cols = ["epe", "epe_median", "epe_rmse", "ae", "ae_median",
                "1pe_pct", "2pe_pct", "3pe_pct", "outlier_3px_5pct", "frames", "valid_pixels"]
summary_table = pd.DataFrame(
    {name: pd.Series(res["benchmark"].get("summary", {})) for name, res in VARIANT_RESULTS.items()}
).reindex(summary_cols)
print("Dense vs sparse benchmark summary")
display(summary_table)

# Bind the active variant for the deep-dive cells below.
benchmark = VARIANT_RESULTS[ACTIVE]["benchmark"]
df = VARIANT_RESULTS[ACTIVE]["df"]
if df.empty:
    print(f"No current benchmark dataframe for active variant '{ACTIVE}'.")
    print("Run event_detector_cpp so logs/moment_flow/.../{dense,sparse} contain DSEC PNGs, then rerun.")
else:
    print(f"\nActive variant '{ACTIVE}' per-frame head/tail:")
    cols = ["frame", "epe", "ae", "1pe_pct", "2pe_pct", "3pe_pct", "valid_pixels", "dt_ms"]
    display(df[cols].head())
    display(df[cols].tail())

## Score Over Time

Look for sudden jumps. A jump often means a timestamp mismatch, dropped replay segment, bad warm start, or a scene-specific failure.

In [ ]:
have_any = any(not res["df"].empty for res in VARIANT_RESULTS.values())
if not have_any:
    print("No current benchmark rows to plot.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    for name, res in VARIANT_RESULTS.items():
        vdf = res["df"]
        if vdf.empty:
            continue
        axes[0].plot(vdf["frame_index"], vdf["epe"], marker="o", label=name)
        axes[1].plot(vdf["frame_index"], vdf["ae"], marker="o", label=name)
        axes[2].plot(vdf["frame_index"], vdf["3pe_pct"], marker="o", label=f"{name} 3PE")
    axes[0].set_ylabel("EPE [px]")
    axes[0].set_title("Per-frame endpoint error (dense vs sparse)")
    axes[0].legend()
    axes[1].set_ylabel("AE [deg]")
    axes[1].legend()
    axes[2].set_ylabel("3PE [%]")
    axes[2].set_xlabel("DSEC frame index")
    axes[2].legend()
    plt.tight_layout()

In [ ]:
cols = ["frame", "epe", "ae", "1pe_pct", "2pe_pct", "3pe_pct", "valid_pixels", "dt_ms"]
if df.empty:
    print("No current benchmark rows for best/worst tables.")
else:
    print("Worst frames by EPE")
    display(df.nlargest(10, "epe")[cols])
    print("Best frames by EPE")
    display(df.nsmallest(10, "epe")[cols])

## DSEC PNG Sanity Checks

DSEC on-disk channel order is RGB = `(flow_x, flow_y, valid)`. OpenCV reads this as BGR, so the raw OpenCV channel 0 should be the valid mask. These checks catch the common channel-order bug immediately.

In [ ]:
def raw_png_stats(path):
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise FileNotFoundError(path)
    flat = raw.reshape(-1, raw.shape[-1])
    return {
        "file": path.name,
        "dtype": str(raw.dtype),
        "shape": raw.shape,
        "bgr_min": flat.min(axis=0).tolist(),
        "bgr_max": flat.max(axis=0).tolist(),
        "cv2_valid_unique": np.unique(raw[..., 0])[:10].tolist(),
        "valid_ratio": float((raw[..., 0] == 1).mean()),
    }

preferred = ["000002.png", "000004.png", "000082.png"]
if not df.empty:
    preferred.extend(df["frame"].head(3).tolist())
sample_files = []
for name in preferred:
    path = PRED_DIR / name
    if path.exists() and path not in sample_files:
        sample_files.append(path)

if not sample_files:
    print("No prediction PNGs available for raw DSEC encoding checks.")
else:
    display(pd.DataFrame([raw_png_stats(path) for path in sample_files]))
    for path in sample_files[:3]:
        frame = load_dsec_png(path)
        print(path.name, "decoded max |flow|", float(np.abs(frame.flow).max()), "valid ratio", float(frame.valid.mean()))

## Visualization Helpers

In [ ]:
def flow_to_rgb(flow, valid=None, clip=None):
    """Middlebury-style HSV visualization for HxWx2 flow."""
    u = flow[..., 0]
    v = flow[..., 1]
    mag = np.sqrt(u * u + v * v)
    if clip is None:
        clip = np.nanpercentile(mag[np.isfinite(mag)], 99) if np.isfinite(mag).any() else 1.0
    clip = max(float(clip), 1e-6)
    ang = np.arctan2(v, u)
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.float32)
    hsv[..., 0] = ((ang + np.pi) / (2 * np.pi) * 179).astype(np.float32)
    hsv[..., 1] = 255
    hsv[..., 2] = np.clip(mag / clip, 0, 1) * 255
    rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    if valid is not None:
        rgb = rgb.copy()
        rgb[~valid] = 0
    return rgb


def load_pair(frame_name):
    gt_path = GT_DIR / frame_name
    pred_path = PRED_DIR / frame_name
    if not pred_path.exists():
        raise FileNotFoundError(f"Missing prediction PNG: {pred_path}")
    gt = load_dsec_png(gt_path)
    pred = load_dsec_png(pred_path)
    # flow_events predictions are sparse: pred.valid marks event-supported pixels.
    # Intersect with it so visualizations/diagnostics match the intersection benchmark.
    mask = (
        gt.valid
        & pred.valid
        & np.isfinite(gt.flow).all(axis=-1)
        & np.isfinite(pred.flow).all(axis=-1)
    )
    diff = pred.flow - gt.flow
    epe = np.linalg.norm(diff, axis=-1)
    return gt, pred, mask, epe


def show_frame(frame_name, error_clip=20.0, flow_clip=None):
    gt, pred, mask, epe = load_pair(frame_name)
    if flow_clip is None:
        both_mag = np.concatenate([
            np.linalg.norm(gt.flow[mask], axis=-1),
            np.linalg.norm(pred.flow[mask], axis=-1),
        ])
        flow_clip = np.percentile(both_mag, 99) if both_mag.size else 1.0
    row = df.loc[df["frame"] == frame_name].iloc[0]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.ravel()
    axes[0].imshow(flow_to_rgb(gt.flow, mask, clip=flow_clip))
    axes[0].set_title(f"GT flow {frame_name}")
    axes[1].imshow(flow_to_rgb(pred.flow, mask, clip=flow_clip))
    axes[1].set_title("MomentFlow prediction")
    im = axes[2].imshow(np.where(mask, epe, np.nan), cmap="turbo", vmin=0, vmax=error_clip)
    axes[2].set_title(f"EPE map, mean={row.epe:.2f}")
    fig.colorbar(im, ax=axes[2], fraction=0.046)
    axes[3].imshow(np.where(mask, gt.flow[..., 0], np.nan), cmap="coolwarm")
    axes[3].set_title("GT u")
    axes[4].imshow(np.where(mask, pred.flow[..., 0], np.nan), cmap="coolwarm")
    axes[4].set_title("Pred u")
    axes[5].hist(epe[mask].ravel(), bins=80, range=(0, error_clip))
    axes[5].set_title("EPE histogram")
    for ax in axes[:5]:
        ax.axis("off")
    plt.tight_layout()
    return gt, pred, mask, epe

## Inspect Worst Frames

All cells from here down operate on the **active** variant (`ACTIVE`, default `"sparse"`). To inspect the other variant, change `ACTIVE` in the config cell and rerun from the load cell. Start with the highest-EPE frame, then change `frame_name` to inspect others.

In [ ]:
if df.empty:
    print("No current benchmark rows to inspect.")
else:
    worst_frame = df.sort_values("epe", ascending=False).iloc[0]["frame"]
    print("Worst frame:", worst_frame)
    _ = show_frame(worst_frame)

In [ ]:
# Optional interactive frame browser. If ipywidgets is unavailable, this falls back to the worst frame.
if df.empty:
    print("No current benchmark rows for the frame browser.")
else:
    try:
        from ipywidgets import Dropdown, interact
        frame_dropdown = Dropdown(options=df["frame"].tolist(), value=worst_frame, description="frame")
        interact(lambda frame: show_frame(frame), frame=frame_dropdown)
    except Exception as exc:
        print("ipywidgets unavailable or disabled:", exc)
        _ = show_frame(worst_frame)

## Sign, Scale, And Axis Diagnostics

If a simple transform greatly improves EPE, the bug is probably not the estimator itself but a convention mismatch: wrong sign, swapped axes, velocity-vs-displacement scaling, or a time-base issue.

In [ ]:
def iter_pairs(frame_names=None):
    if df.empty:
        return
    names = frame_names if frame_names is not None else df["frame"].tolist()
    for name in names:
        pred_path = PRED_DIR / name
        if pred_path.exists():
            yield name, *load_pair(name)

variants = {
    "identity": lambda f: f,
    "neg_xy": lambda f: -f,
    "flip_x": lambda f: np.dstack([-f[..., 0], f[..., 1]]),
    "flip_y": lambda f: np.dstack([f[..., 0], -f[..., 1]]),
    "swap_xy": lambda f: np.dstack([f[..., 1], f[..., 0]]),
    "swap_neg_xy": lambda f: -np.dstack([f[..., 1], f[..., 0]]),
}


def variant_score(transform):
    dot = 0.0
    norm = 0.0
    raw_epe_sum = 0.0
    count = 0
    for _, gt, pred, mask, _ in iter_pairs():
        p = transform(pred.flow)[mask].reshape(-1, 2).astype(np.float64)
        g = gt.flow[mask].reshape(-1, 2).astype(np.float64)
        dot += float((p * g).sum())
        norm += float((p * p).sum())
        raw_epe_sum += float(np.linalg.norm(p - g, axis=-1).sum())
        count += p.shape[0]
    if count == 0:
        return {"raw_epe": np.nan, "best_scalar": np.nan, "scaled_epe": np.nan}
    scale = dot / max(norm, 1e-12)
    scaled_epe_sum = 0.0
    for _, gt, pred, mask, _ in iter_pairs():
        p = transform(pred.flow)[mask].reshape(-1, 2).astype(np.float64) * scale
        g = gt.flow[mask].reshape(-1, 2).astype(np.float64)
        scaled_epe_sum += float(np.linalg.norm(p - g, axis=-1).sum())
    return {
        "raw_epe": raw_epe_sum / count,
        "best_scalar": scale,
        "scaled_epe": scaled_epe_sum / count,
    }

if df.empty:
    variant_df = pd.DataFrame()
    print("No current benchmark rows for sign/scale diagnostics.")
else:
    variant_df = pd.DataFrame([
        {"variant": name, **variant_score(fn)}
        for name, fn in variants.items()
    ]).sort_values("scaled_epe")
    display(variant_df)

In [ ]:
# Fit a global 2x2 linear map pred @ A ~= gt from a random pixel sample.
if df.empty:
    print("No current benchmark rows for the global linear fit.")
else:
    rng = np.random.default_rng(0)
    P_samples = []
    G_samples = []
    for name, gt, pred, mask, _ in iter_pairs():
        ys, xs = np.where(mask)
        if ys.size == 0:
            continue
        take = rng.choice(ys.size, size=min(6000, ys.size), replace=False)
        p = pred.flow[ys[take], xs[take]].astype(np.float64)
        g = gt.flow[ys[take], xs[take]].astype(np.float64)
        P_samples.append(p)
        G_samples.append(g)
    if not P_samples:
        print("No valid pixels available for the global linear fit.")
    else:
        P = np.vstack(P_samples)
        G = np.vstack(G_samples)
        A, *_ = np.linalg.lstsq(P, G, rcond=None)  # P @ A ~= G
        raw_sample_epe = np.linalg.norm(P - G, axis=1).mean()
        fit_sample_epe = np.linalg.norm(P @ A - G, axis=1).mean()
        print("A maps [pred_u, pred_v] to [gt_u, gt_v]:")
        display(pd.DataFrame(A, index=["pred_u", "pred_v"], columns=["gt_u", "gt_v"]))
        print(f"sample raw EPE: {raw_sample_epe:.3f}")
        print(f"sample fitted-linear EPE: {fit_sample_epe:.3f}")

Interpretation tips:

- `neg_xy` much better than `identity`: saved image-flow sign is probably flipped.
- `swap_xy` much better: x/y channel or coordinate convention is wrong.
- `best_scalar` near `0.1`, `10`, `100`, or `0.01`: likely velocity/displacement or timestamp scaling issue.
- A fitted 2x2 matrix that greatly improves EPE but is not close to diagonal suggests mixed axes or a rotation-like convention bug.

## Spatial Error Bias

A spatially concentrated error can point to rectification mismatch, border effects, tile-grid artifacts, or parts of the image with low event support.

In [ ]:
sum_epe = None
count = None
sum_gt_mag = None
sum_pred_mag = None
for _, gt, pred, mask, epe in iter_pairs():
    if sum_epe is None:
        sum_epe = np.zeros_like(epe, dtype=np.float64)
        count = np.zeros_like(epe, dtype=np.float64)
        sum_gt_mag = np.zeros_like(epe, dtype=np.float64)
        sum_pred_mag = np.zeros_like(epe, dtype=np.float64)
    sum_epe[mask] += epe[mask]
    count[mask] += 1
    sum_gt_mag[mask] += np.linalg.norm(gt.flow[mask], axis=-1)
    sum_pred_mag[mask] += np.linalg.norm(pred.flow[mask], axis=-1)

if sum_epe is None:
    print("No current prediction/GT pairs for spatial bias maps.")
else:
    mean_epe = np.divide(sum_epe, count, out=np.full_like(sum_epe, np.nan), where=count > 0)
    mean_gt_mag = np.divide(sum_gt_mag, count, out=np.full_like(sum_gt_mag, np.nan), where=count > 0)
    mean_pred_mag = np.divide(sum_pred_mag, count, out=np.full_like(sum_pred_mag, np.nan), where=count > 0)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, img, title, vmax in [
        (axes[0], mean_epe, "Mean EPE", np.nanpercentile(mean_epe, 98)),
        (axes[1], count, "GT valid count", np.nanmax(count)),
        (axes[2], mean_gt_mag, "Mean GT magnitude", np.nanpercentile(mean_gt_mag, 98)),
        (axes[3], mean_pred_mag, "Mean pred magnitude", np.nanpercentile(mean_pred_mag, 98)),
    ]:
        im = ax.imshow(img, cmap="turbo", vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()

## Component And Magnitude Correlation

In [ ]:
# Sample pixels for scatter plots.
rng = np.random.default_rng(1)
S_pred = []
S_gt = []
for _, gt, pred, mask, _ in iter_pairs():
    ys, xs = np.where(mask)
    if ys.size == 0:
        continue
    take = rng.choice(ys.size, size=min(2500, ys.size), replace=False)
    S_pred.append(pred.flow[ys[take], xs[take]])
    S_gt.append(gt.flow[ys[take], xs[take]])

if not S_pred:
    print("No current prediction/GT pairs for correlation scatter plots.")
else:
    S_pred = np.vstack(S_pred)
    S_gt = np.vstack(S_gt)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(S_gt[:, 0], S_pred[:, 0], s=2, alpha=0.15)
    axes[0].set_xlabel("GT u")
    axes[0].set_ylabel("Pred u")
    axes[0].set_title("u correlation")
    axes[1].scatter(S_gt[:, 1], S_pred[:, 1], s=2, alpha=0.15, color="tab:orange")
    axes[1].set_xlabel("GT v")
    axes[1].set_ylabel("Pred v")
    axes[1].set_title("v correlation")
    gt_mag = np.linalg.norm(S_gt, axis=1)
    pred_mag = np.linalg.norm(S_pred, axis=1)
    axes[2].scatter(gt_mag, pred_mag, s=2, alpha=0.15, color="tab:green")
    axes[2].set_xlabel("GT magnitude")
    axes[2].set_ylabel("Pred magnitude")
    axes[2].set_title("magnitude correlation")
    for ax in axes:
        lo, hi = ax.get_xlim()
        ylo, yhi = ax.get_ylim()
        mn, mx = min(lo, ylo), max(hi, yhi)
        ax.plot([mn, mx], [mn, mx], "k--", linewidth=1)
    plt.tight_layout()

## Timestamp And Output Completeness Checks

In [ ]:
gt_files = sorted(GT_DIR.glob("*.png"))
pred_files = sorted(PRED_DIR.glob("*.png"))
expected = [p.name for p in gt_files]
actual = [p.name for p in pred_files]
missing = sorted(set(expected) - set(actual))
extra = sorted(set(actual) - set(expected))
print(f"GT PNGs: {len(gt_files)}  Pred PNGs: {len(pred_files)}")
print("missing:", missing[:20])
print("extra:", extra[:20])
if df.empty:
    print("Timestamp dt range unavailable until the benchmark has current rows.")
else:
    print("Timestamp dt range [ms]:", df["dt_ms"].min(), df["dt_ms"].max())
    display(df[["frame", "from_us", "to_us", "dt_ms", "valid_pixels", "epe"]].head(10))

## Next Things To Try

Use the tables above to pick the next experiment:

1. If output completeness reports `Pred PNGs: 0`, rerun `event_detector_cpp` and check `flow_save_enabled`, `flow_save_output_dir`, the timestamp schedule path, and whether the node is being started before replay.
2. If sign/axis diagnostics improve a lot, fix the C++ save convention before tuning the algorithm.
3. If `best_scalar` is far from `1`, inspect whether the estimator is saved as px/s, px/ms, or displacement and whether `dt_us` is correct.
4. If errors are spatially localized, compare event rectification and GT rectified view assumptions.
5. If only a few frames dominate, open those frames with `show_frame("000xxx.png")` and inspect the flow/error maps.
6. If prediction magnitude is consistently too small, tune regularization/fallback/smoothing only after confirming timestamps and units are correct.